# Runda 8: fondägare och analytiker från LSEG (version 4)

1. Lägg denna notebook och `runda8_input.csv` i mappen `runda8` i CodeBook. Ersätt den gamla notebooken.
2. Kör allt: **Run → Run All Cells**. Det tar troligen 30–60 minuter.
3. Ladda ner de fyra filerna och ladda upp dem till Claude:
   `runda8_fondagare.csv.gz`, `runda8_analytiker.csv.gz`, `runda8_logg.txt`, `runda8_falttest.txt`

Avbryts körningen: kör alla celler igen. Den fortsätter där den slutade.

In [ ]:
import time
import pandas as pd
try:
    import lseg.data as ld
except Exception:
    import refinitiv.data as ld
ld.open_session()
E = pd.read_csv('runda8_input.csv')
EV = E[E.kind == 'event'].copy()
RICS = sorted(set(E.ric.dropna()))
LOGG = []
def logg(s):
    LOGG.append(str(s)); print(s)
def spara_logg():
    with open('runda8_logg.txt', 'w') as f:
        f.write('\n'.join(LOGG))
logg(f'session öppen: {len(EV)} händelser, {len(RICS)} aktier totalt')

In [ ]:
# ---------- 1. Fälttest: vilka fältnamn fungerar? (cirka 3 minuter)
TEST = ['VOLVa.ST', 'BILIa.ST', 'BOSS.ST^D08']
FOND = ['TR.FundPortfolioName', 'TR.FundInvestorType', 'TR.FundAdjShrsHeld', 'TR.FundSharesHeld', 'TR.FundPctOfShrsOutHeld',
        'TR.FundPctOfSharesOutHeld', 'TR.FundHoldingsDate', 'TR.FundHoldingDate', 'TR.FundAddrCountry', 'TR.FundPermID',
        'TR.FundInvestorID', 'TR.FundParentName', 'TR.FundManagerName', 'TR.FundSharesHeldValue', 'TR.FundValueHeld']
INV = ['TR.InvestorFullName', 'TR.InvestorType', 'TR.SharesHeld', 'TR.PctOfSharesOutHeld', 'TR.HoldingsDate', 'TR.InvAddrCountry', 'TR.InvParentType']
ANA = ['TR.EPSNumIncEstimates', 'TR.EPSNumberofEstimates', 'TR.EPSNumberOfEstimates', 'TR.NumberOfAnalysts',
       'TR.RecNoOfRecommendations', 'TR.NumOfRecommendations', 'TR.RevenueNumIncEstimates']
def prova(ric, falt, par):
    try:
        d = ld.get_data(universe=[ric], fields=[falt], parameters=par)
        kol = [c for c in d.columns if c.lower() != 'instrument']
        n = int(d[kol].notna().any(axis=1).sum()) if kol else 0
        ex = d[kol[0]].dropna().astype(str).head(2).tolist() if (kol and n) else []
        return n, ex, ''
    except Exception as e:
        return 0, [], str(e)[:90]
FUNKAR = {'fond': [], 'inv': [], 'ana': []}
for grupp, lista, par in [('fond', FOND, {'SDate': '2015-05-29'}), ('inv', INV, {'SDate': '2015-05-29'}),
                          ('ana', ANA, {'Period': 'FY1', 'SDate': '2010-01-01', 'EDate': '2011-12-31', 'Frq': 'M'})]:
    for f in lista:
        res = [prova(r, f, par) for r in TEST]
        ok = sum(n > 0 for n, _, _ in res)
        logg(f'{grupp} {f}: värden för {ok} av 3 aktier; rader {[n for n, _, _ in res]}; exempel {res[0][1] or res[1][1]}; fel {[e for _, _, e in res if e][:1]}')
        if ok >= 2: FUNKAR[grupp].append(f)
for r in TEST[:2]:
    for start, slut in [('2015-01-01', '2015-06-30'), ('2025-01-01', '2025-06-30')]:
        try:
            h = ld.news.get_headlines(query=f'R:{r}', start=start, end=slut, count=100)
            logg(f'nyheter {r} {start}: {0 if h is None else len(h)} rubriker')
        except Exception as e:
            logg(f'nyheter {r} {start}: FEL {str(e)[:90]}')
logg(f'FUNGERAR: {FUNKAR}')
spara_logg()
with open('runda8_falttest.txt', 'w') as f:
    f.write('\n'.join(LOGG))

In [ ]:
# ---------- 2. Fondägare per händelse: före beskedet och efter ändringen (huvuddelen, 20-50 minuter)
FOND_OK = FUNKAR['fond'] or FUNKAR['inv']          # fondnivå om det fungerar, annars investerarnivå
NIVA = 'fond' if FUNKAR['fond'] else 'investerare'
UT = 'runda8_fondagare.csv.gz'
try:
    GAMMAL = pd.read_csv(UT); DELAR = [GAMMAL]; KLARA = set(zip(GAMMAL.event_id, GAMMAL.snap))
    logg(f'fortsätter: {len(KLARA)} ögonblicksbilder redan klara')
except Exception:
    DELAR, KLARA = [], set()
def spara_agare():
    if DELAR:
        pd.concat(DELAR, ignore_index=True).to_csv(UT, index=False)
logg(f'hämtar fondägare på {NIVA}nivå med {FOND_OK}')
if FOND_OK:
    idag = pd.Timestamp.today()
    for i, x in enumerate(EV.itertuples()):
        for snap, d in [('fore', pd.Timestamp(x.AD) - pd.Timedelta(days=1)), ('efter', pd.Timestamp(x.ED) + pd.Timedelta(days=45))]:
            if (x.event_id, snap) in KLARA or d > idag: continue
            df, fel = None, ''
            for forsok in range(3):
                try:
                    df = ld.get_data(universe=[x.ric], fields=FOND_OK, parameters={'SDate': d.strftime('%Y-%m-%d')}); break
                except Exception as e:
                    fel = str(e)[:90]; time.sleep(3)
            if df is None or len(df) == 0:
                df = pd.DataFrame({'tom': [True]}); logg(f'{x.event_id} {snap}: inga rader {fel}')
            df['event_id'] = x.event_id; df['ric'] = x.ric; df['snap'] = snap; df['asof'] = str(d.date()); df['niva'] = NIVA
            DELAR.append(df); KLARA.add((x.event_id, snap))
        if i % 25 == 0:
            spara_agare(); spara_logg(); print(f'{i} av {len(EV)} händelser klara (sparat)')
    spara_agare()
    logg(f'fondägare klart: {len(KLARA)} ögonblicksbilder')
spara_logg()

In [ ]:
# ---------- 3. Analytikertäckning per månad, alla händelseaktier och kontrollaktier (cirka 5-10 minuter)
UT2 = 'runda8_analytiker.csv.gz'
try:
    G2 = pd.read_csv(UT2); DELAR2 = [G2]; KLARA_R = set(G2['Instrument'].astype(str))
except Exception:
    DELAR2, KLARA_R = [], set()
KVAR = [r for r in RICS if r not in KLARA_R]          # bara aktier som inte redan är hämtade
logg(f'analytiker: {len(KLARA_R)} aktier redan hämtade, {len(KVAR)} kvar')
if FUNKAR['ana']:
    A = FUNKAR['ana'][0]
    for k in range(0, len(KVAR), 40):
        b = KVAR[k:k + 40]; df = None
        for forsok in range(3):
            try:
                df = ld.get_data(universe=b, fields=[A, A + '.date'],
                                 parameters={'Period': 'FY1', 'SDate': '2006-01-01', 'EDate': '2026-10-31', 'Frq': 'M'}); break
            except Exception as e:
                logg(f'analytiker del {k}: FEL {str(e)[:90]}'); time.sleep(3)
        if df is not None:
            df['falt'] = A; df['batch'] = k; DELAR2.append(df)
            pd.concat(DELAR2, ignore_index=True).to_csv(UT2, index=False)
        print(f'analytiker: {min(k + 40, len(KVAR))} av {len(KVAR)} kvarvarande aktier')
    logg(f'analytiker klart med {A}')
else:
    logg('inget analytikerfält fungerade')
spara_logg()
print('Klart. Ladda ner runda8_fondagare.csv.gz, runda8_analytiker.csv.gz, runda8_logg.txt och runda8_falttest.txt och ladda upp dem till Claude.')